TIENDA ONLINE

-Se incorpora el loop, "agregar_listado_productos" para subir de forma masiva un listado. No modifica precio, solo suma cantidades.
-Se incorpora clientes como atributo a la clase.
-Se incorpora la función "agregar_cliente".
-Se incorpora la función "ver_clientes" y se importa la librería "tabulete".
-Se incorpora la función "realizar_compra".

In [ ]:
class TiendaOnLine:
    
    def __init__(self, nombre):
        self.nombre = nombre
        self.inventario = []                                # Lista de diccionarios: cada dict es {"nombre": str, "precio": float, "cantidad": int}
        self.ventas_totales = 0
        self.clientes = {}

    
    def agregar_producto(self, nombre, precio, cantidad):
        # Validaciones
        if not nombre or not precio or not cantidad:
            print(f"Debes ingresar nombre, precio y cantidad del producto.\nEjecuta nuevamente.")
            return
        if precio <= 0:                                     #posibilidad de mejora, imposibilidad de ingreso de string.
            print("Debes ingresar un precio mayor a 0.")
            return
        if cantidad <= 0 or not isinstance(cantidad,int):   #posibilidad de mejora, imposibilidad de ingreso de string.
            print("Debes ingresar un número entero, mayor a 0.")
            return

        # Busqueda en el inventario
        for item in self.inventario:
            if item["nombre"] == nombre.title().strip():
                item["cantidad"] += cantidad
                print(f"Se actualizaron las unidades del producto '{nombre.title()}'. Hay {item['cantidad']:,.0f} unidad(es) disponible(s).") #title para corregir error de usuario.
                return
        
        # Si no existe, agregar nuevo producto
        nuevo_producto = {"nombre": nombre.title().strip(), "precio": precio, "cantidad": cantidad}
        self.inventario.append(nuevo_producto)
        print(f"El producto '{nombre.title().strip()}' fue agregado al inventario.")  #sin el title(), replica el nombre ingresado para corroborar con el usuario que fuera error de tipeo.
        print(self.inventario)

           
    def agregar_listado_productos(self,listado_productos):  #""" AGREGAR LISTA DE PRODUCTOS """

        for prod in listado_productos:
            self.agregar_producto(prod['nombre'],prod['precio'],prod['cantidad'])
    
    def ver_inventario(self):
        if not self.inventario:
            print("El inventario está vacío.")
            #return
        
        for producto in self.inventario:
            print(f"Nombre: {producto['nombre']}, Precio: ${producto['precio']:,.2f}, Cantidad: {producto['cantidad']:,.0f}") #corregido nombre-producto.

    def buscar_producto(self,nombre):
        
        prod = nombre.title().strip()   #renombrado nombre en prod para acceder al formato en que se registra en el diccionario.
        encontrado = False

        for producto in self.inventario:
            if prod == producto['nombre']:
                print(f"El producto '{prod}' se encuentra en el inventario.\nNombre: {producto['nombre']}, Precio: ${producto['precio']:,.2f}, Cantidad: {producto['cantidad']:,.0f}")
                encontrado = True

        if not encontrado:
            print(f"El producto '{prod}' no se encuentra en el inventario.")

    def actualizar_stock(self,nombre,cantidad):

        if not self.inventario:
            print(f"Debes ingresar un nombre de producto válido.")
            return

        if not isinstance(cantidad,int):
            print(f"No puedes indicar cantidades con decimales, deben ser números enteros.")
            return

        if cantidad == 0:
            print("La cantidad ingresada es 0(cero). No se modificó el stock.")

        for producto in self.inventario:

            if cantidad >0:
                if nombre.title().strip() == producto['nombre']:
                    producto['cantidad']+=cantidad
                    print(f"Se actualizó el stock:\nNombre: {producto['nombre']}, Precio: ${producto['precio']:,.2f}, Cantidad: {producto['cantidad']:,.0f}")
                    break

            else:
                if nombre.title().strip() == producto['nombre'] and (- cantidad) > producto['cantidad']:    #evita variación de stock menor a cero.
                    print(f"Sólo se pudo descontar {producto['cantidad']} unidad(es). El stock del producto está en 0(cero).")
                    producto['cantidad'] = 0
                    print(f"Nombre: {producto['nombre']}, Precio: ${producto['precio']:,.2f}, Cantidad: {producto['cantidad']:,.0f}")
                    break

        else:
            print(f"El producto '{nombre.title().strip()}', no se encuentra en el inventario.")

    def eliminar_producto(self,nombre):
        
        if not nombre:
            print("Debes ingresar un nombre de producto.")

        for producto in self.inventario:
            if nombre.title().strip() == producto['nombre']:
                self.inventario.remove(producto)
                print(f"El producto '{nombre}', se ha eliminado del inventario.")
                break
        else:
            print(f"El producto '{nombre.title().strip()}', no se encuentra en el inventario.")

    def calcular_valor_inventario(self):
        total_inventario=0
        for producto in self.inventario:
            valor=producto['precio']*producto['cantidad']
            total_inventario+=valor
        print(f"El valor total del inventario es de ${total_inventario:,.2f}.\n")

### BONUS ###


    def agregar_cliente(self,nombre,correo):

        nom=nombre.title().strip()  #para homogeneizar nomenclatura.

        if not nombre or not correo:
            print("Debes ingresar un nombre de cliente y mail.")
            return    
      
        for cliente in self.clientes:
            
            if nom == cliente:
                print(f"El cliente {nom}, ya se encuentra registrado.")
                break 
                     
        else:
            self.clientes[nom] = {"email":correo, "compras":[]}
            print(self.clientes)
           
    def ver_clientes(self):

        from tabulate import tabulate   #se importa libreria tabulate para salida.
        
        data = [[nombre, datos['email'], len(datos['compras'])]
        for nombre, datos in self.clientes.items()]
        
        print("\n" + tabulate(data, headers=["Cliente", "Email", "Compras"], tablefmt="rounded_grid", colalign=("left","left","left")))


    def realizar_compra(self, nombre_cliente):
    
        nombre_cliente = nombre_cliente.title().strip()     #valida cliente registrado.
        if nombre_cliente not in self.clientes:
            print(f"El cliente '{nombre_cliente}' no está registrado.")
            return

        if not self.inventario:                             #valida inventario no vacío.
            print("El inventario está vacío. No se pueden realizar compras.")
            return

        print("\nInventario Disponible:\n\n")                 #muestra inventario al cliente.
        for i, prod in enumerate(self.inventario, start=1):
            print(f"{i}. {prod['nombre']} - ${prod['precio']:.2f} (stock: {prod['cantidad']})")

    
        carrito = {}                            #carrito es un diccionario --> {nombre_producto: cantidad_total}

        while True:
            
            print("\nCarrito:")               #carrito actual.

            if not carrito:
                print("(vacío)")

            else:
                total_parcial = 0
                for nombre, cantidad in carrito.items():                #busca precio actual del producto en inventario
                    precio = next(prod['precio'] for prod in self.inventario if prod['nombre'] == nombre)
                    subtotal = precio * cantidad
                    total_parcial += subtotal
                    print(f"{nombre}x{cantidad} = ${subtotal:,.2f}")

                print(f"\nTotal: ${total_parcial:,.2f}\n")

            
            comprar_producto = input("\n¿Qué producto quieres comprar? (Podes ingresar el número, nombre o 'salir'): ").strip()       # pide producto.

            if comprar_producto.lower() == 'salir':
                break
            
            producto = None                         # buscador de producto por número o nombre. 
            if comprar_producto.isdigit():
                indice_prod = int(comprar_producto) - 1
                if 0 <= indice_prod < len(self.inventario):
                    producto = self.inventario[indice_prod]
            else:
                nombre_prod = comprar_producto.title().strip()
                for prod in self.inventario:
                    if prod['nombre'] == nombre_prod:
                        producto = prod
                        break

            if not producto:                        #si no encuentra producto.
                print("Producto no encontrado. Puedes consultar nuevamente el inventario.")
                continue

            try:                                                    #pide cantidad.
                comprar_cantidad = int(input(f"¿Cuántas unidades de '{producto['nombre']}' quieres comprar? "))  #A QUE LLAMA VER
                if comprar_cantidad <= 0:
                    print("La cantidad debe ser un número positivo.")
                    continue
            except ValueError:
                print("Por favor, ingresa un número entero válido.")
                continue

   
            if comprar_cantidad > producto['cantidad']:             #verifica stock.
                print(f"No contamos con stock suficiente. Solo hay {producto['cantidad']} unidades.")
                continue


            nombre_prod = producto['nombre']                #se agrega al carrito.
            carrito[nombre_prod] = carrito.get(nombre_prod, 0) + comprar_cantidad
            print(f" {comprar_cantidad} x '{nombre_prod}' agregado al carrito.") #VER SI ELIMINO O NO EL ESPACIO SEGUN PRINT.

        
        if not carrito:                     # verifica si se agregaron productos. #VER CUANDO ARROJA ESTE MENSAJE, CON 'SALIR'?
            print("No se agregaron productos. Compra cancelada.")
            return

        print("\nResumen de Compra:")            # subtotal y confirmación de compra.
        total_compra = 0
        for nombre, cant in carrito.items():
            precio = next(p['precio'] for p in self.inventario if p['nombre'] == nombre)
            subtotal = precio * cant
            total_compra += subtotal
            print(f"{nombre} x{cant} = ${subtotal:,.2f}")
        print(f"Total a Pagar: ${total_compra:,.2f}")

        confirmar = input("\n¿Confirmar compra? (s/n): ").strip().lower()
        if confirmar != 's':
            print("\nCompra cancelada.")                          #no se modifica el inventario.
            return

        for nombre, cant in carrito.items():               #procesa la compra (afecta stock, actualiza ventas, registra historial)
            for prod in self.inventario:
                if prod['nombre'] == nombre:
                    prod['cantidad'] -= cant
                    monto = prod['precio'] * cant
                    self.ventas_totales += monto
                    self.clientes[nombre_cliente]['compras'].append({'producto': nombre,'cantidad': cant, 'monto': monto})
                    break

        print("\n¡Tu compra se realizó exitosamente!")
        print(f"\nVentas totales acumuladas: ${self.ventas_totales:,.2f}")   







In [702]:
farma=TiendaOnLine("Farma_Salud")

In [703]:
farma.agregar_listado_productos([{'nombre': 'Vitamina C 1000mg (30 tabs)', 'precio': 12.50, 'cantidad': 150},
    {'nombre': 'Gel antibacterial (500ml)', 'precio': 8.99, 'cantidad': 300},
    {'nombre': 'Termómetro digital', 'precio': 15.90, 'cantidad': 80},
    {'nombre': 'Mascarillas KN95 (10 uds)', 'precio': 9.99, 'cantidad': 500},
    {'nombre': 'Jabón de avena hipoalergénico', 'precio': 4.50, 'cantidad': 200},
    {'nombre': 'Colágeno hidrolizado (300g)', 'precio': 29.99, 'cantidad': 60},
    {'nombre': 'Vendas elásticas (2 uds)', 'precio': 7.20, 'cantidad': 120},
    {'nombre': 'Probióticos 50 mil millones', 'precio': 34.90, 'cantidad': 45},
    {'nombre': 'Nebulizador portátil', 'precio': 49.99, 'cantidad': 25},
    {'nombre': 'Pillbox organizador semanal', 'precio': 6.75, 'cantidad': 180}]
)

El producto 'Vitamina C 1000Mg (30 Tabs)' fue agregado al inventario.
[{'nombre': 'Vitamina C 1000Mg (30 Tabs)', 'precio': 12.5, 'cantidad': 150}]
El producto 'Gel Antibacterial (500Ml)' fue agregado al inventario.
[{'nombre': 'Vitamina C 1000Mg (30 Tabs)', 'precio': 12.5, 'cantidad': 150}, {'nombre': 'Gel Antibacterial (500Ml)', 'precio': 8.99, 'cantidad': 300}]
El producto 'Termómetro Digital' fue agregado al inventario.
[{'nombre': 'Vitamina C 1000Mg (30 Tabs)', 'precio': 12.5, 'cantidad': 150}, {'nombre': 'Gel Antibacterial (500Ml)', 'precio': 8.99, 'cantidad': 300}, {'nombre': 'Termómetro Digital', 'precio': 15.9, 'cantidad': 80}]
El producto 'Mascarillas Kn95 (10 Uds)' fue agregado al inventario.
[{'nombre': 'Vitamina C 1000Mg (30 Tabs)', 'precio': 12.5, 'cantidad': 150}, {'nombre': 'Gel Antibacterial (500Ml)', 'precio': 8.99, 'cantidad': 300}, {'nombre': 'Termómetro Digital', 'precio': 15.9, 'cantidad': 80}, {'nombre': 'Mascarillas Kn95 (10 Uds)', 'precio': 9.99, 'cantidad': 500

In [704]:
farma.ver_inventario()

Nombre: Vitamina C 1000Mg (30 Tabs), Precio: $12.50, Cantidad: 150
Nombre: Gel Antibacterial (500Ml), Precio: $8.99, Cantidad: 300
Nombre: Termómetro Digital, Precio: $15.90, Cantidad: 80
Nombre: Mascarillas Kn95 (10 Uds), Precio: $9.99, Cantidad: 500
Nombre: Jabón De Avena Hipoalergénico, Precio: $4.50, Cantidad: 200
Nombre: Colágeno Hidrolizado (300G), Precio: $29.99, Cantidad: 60
Nombre: Vendas Elásticas (2 Uds), Precio: $7.20, Cantidad: 120
Nombre: Probióticos 50 Mil Millones, Precio: $34.90, Cantidad: 45
Nombre: Nebulizador Portátil, Precio: $49.99, Cantidad: 25
Nombre: Pillbox Organizador Semanal, Precio: $6.75, Cantidad: 180


In [705]:
print(type(farma.clientes))
print(len(farma.clientes))
print(farma.clientes)

<class 'dict'>
0
{}


In [706]:
farma.agregar_cliente("","prueba@email.com")

Debes ingresar un nombre de cliente y mail.


In [707]:
farma.agregar_cliente("Luisa Trujillo","bea@gmail.com")

{'Luisa Trujillo': {'email': 'bea@gmail.com', 'compras': []}}


In [708]:
print(farma.clientes)

{'Luisa Trujillo': {'email': 'bea@gmail.com', 'compras': []}}


In [709]:
farma.ver_clientes()


╭────────────────┬───────────────┬───────────╮
│ Cliente        │ Email         │ Compras   │
├────────────────┼───────────────┼───────────┤
│ Luisa Trujillo │ bea@gmail.com │ 0         │
╰────────────────┴───────────────┴───────────╯


In [710]:
farma.agregar_cliente("Crono Bunge", "crono@hotmail.com")

{'Luisa Trujillo': {'email': 'bea@gmail.com', 'compras': []}, 'Crono Bunge': {'email': 'crono@hotmail.com', 'compras': []}}


In [711]:
farma.realizar_compra("Crono Bunge")


Inventario Disponible:


1. Vitamina C 1000Mg (30 Tabs) - $12.50 (stock: 150)
2. Gel Antibacterial (500Ml) - $8.99 (stock: 300)
3. Termómetro Digital - $15.90 (stock: 80)
4. Mascarillas Kn95 (10 Uds) - $9.99 (stock: 500)
5. Jabón De Avena Hipoalergénico - $4.50 (stock: 200)
6. Colágeno Hidrolizado (300G) - $29.99 (stock: 60)
7. Vendas Elásticas (2 Uds) - $7.20 (stock: 120)
8. Probióticos 50 Mil Millones - $34.90 (stock: 45)
9. Nebulizador Portátil - $49.99 (stock: 25)
10. Pillbox Organizador Semanal - $6.75 (stock: 180)

Carrito:
(vacío)
 4 x 'Vitamina C 1000Mg (30 Tabs)' agregado al carrito.

Carrito:
Vitamina C 1000Mg (30 Tabs)x4 = $50.00

Total: $50.00

 9 x 'Colágeno Hidrolizado (300G)' agregado al carrito.

Carrito:
Vitamina C 1000Mg (30 Tabs)x4 = $50.00
Colágeno Hidrolizado (300G)x9 = $269.91

Total: $319.91

Producto no encontrado. Puedes consultar nuevamente el inventario.

Carrito:
Vitamina C 1000Mg (30 Tabs)x4 = $50.00
Colágeno Hidrolizado (300G)x9 = $269.91

Total: $319.91


In [712]:
farma.realizar_compra("Luisa Trujillo")


Inventario Disponible:


1. Vitamina C 1000Mg (30 Tabs) - $12.50 (stock: 146)
2. Gel Antibacterial (500Ml) - $8.99 (stock: 300)
3. Termómetro Digital - $15.90 (stock: 80)
4. Mascarillas Kn95 (10 Uds) - $9.99 (stock: 500)
5. Jabón De Avena Hipoalergénico - $4.50 (stock: 200)
6. Colágeno Hidrolizado (300G) - $29.99 (stock: 51)
7. Vendas Elásticas (2 Uds) - $7.20 (stock: 120)
8. Probióticos 50 Mil Millones - $34.90 (stock: 45)
9. Nebulizador Portátil - $49.99 (stock: 25)
10. Pillbox Organizador Semanal - $6.75 (stock: 180)

Carrito:
(vacío)
Producto no encontrado. Puedes consultar nuevamente el inventario.

Carrito:
(vacío)
Producto no encontrado. Puedes consultar nuevamente el inventario.

Carrito:
(vacío)
Producto no encontrado. Puedes consultar nuevamente el inventario.

Carrito:
(vacío)
Producto no encontrado. Puedes consultar nuevamente el inventario.

Carrito:
(vacío)
Producto no encontrado. Puedes consultar nuevamente el inventario.

Carrito:
(vacío)
 6 x 'Colágeno Hidrolizado (30

In [713]:
farma.ver_inventario()

Nombre: Vitamina C 1000Mg (30 Tabs), Precio: $12.50, Cantidad: 146
Nombre: Gel Antibacterial (500Ml), Precio: $8.99, Cantidad: 300
Nombre: Termómetro Digital, Precio: $15.90, Cantidad: 79
Nombre: Mascarillas Kn95 (10 Uds), Precio: $9.99, Cantidad: 500
Nombre: Jabón De Avena Hipoalergénico, Precio: $4.50, Cantidad: 200
Nombre: Colágeno Hidrolizado (300G), Precio: $29.99, Cantidad: 45
Nombre: Vendas Elásticas (2 Uds), Precio: $7.20, Cantidad: 118
Nombre: Probióticos 50 Mil Millones, Precio: $34.90, Cantidad: 45
Nombre: Nebulizador Portátil, Precio: $49.99, Cantidad: 25
Nombre: Pillbox Organizador Semanal, Precio: $6.75, Cantidad: 180


In [714]:
farma.realizar_compra("Luisa Trujillo")


Inventario Disponible:


1. Vitamina C 1000Mg (30 Tabs) - $12.50 (stock: 146)
2. Gel Antibacterial (500Ml) - $8.99 (stock: 300)
3. Termómetro Digital - $15.90 (stock: 79)
4. Mascarillas Kn95 (10 Uds) - $9.99 (stock: 500)
5. Jabón De Avena Hipoalergénico - $4.50 (stock: 200)
6. Colágeno Hidrolizado (300G) - $29.99 (stock: 45)
7. Vendas Elásticas (2 Uds) - $7.20 (stock: 118)
8. Probióticos 50 Mil Millones - $34.90 (stock: 45)
9. Nebulizador Portátil - $49.99 (stock: 25)
10. Pillbox Organizador Semanal - $6.75 (stock: 180)

Carrito:
(vacío)
No contamos con stock suficiente. Solo hay 146 unidades.

Carrito:
(vacío)
 2 x 'Colágeno Hidrolizado (300G)' agregado al carrito.

Carrito:
Colágeno Hidrolizado (300G)x2 = $59.98

Total: $59.98

 1 x 'Nebulizador Portátil' agregado al carrito.

Carrito:
Colágeno Hidrolizado (300G)x2 = $59.98
Nebulizador Portátilx1 = $49.99

Total: $109.97


Resumen de Compra:
Colágeno Hidrolizado (300G) x2 = $59.98
Nebulizador Portátil x1 = $49.99
Total a Pagar: $109